# Tutorial 12 Deep Reinforcement Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_12_Deep_Reinforcement_Learning/tutorial_12_pytorch.ipynb)

Learn discrete and continuous control with a stochastic actor–critic, DDPG, and a constrained toy materials environment. Compare trained agents against baselines using separate evaluation seeds.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** PyTorch, gradients, neural networks, and basic probability.

**Resources:** CPU-friendly default; several minutes, no dataset download, API key, ROM, or Drive mount. Gymnasium classic-control rendering is included. A single training seed is a teaching run, not a robustness benchmark.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals and scope

1. Follow observation → action → reward → next observation.
2. Distinguish true termination from a time-limit truncation, including in bootstrap targets.
3. Train a discrete stochastic actor with a learned value baseline on CartPole-v1.
4. Implement continuous DDPG with replay, frozen targets, exploration noise, and soft updates on Pendulum-v1.
5. Reuse DDPG for a **synthetic** materials composition task on the simplex; compare to an analytic controller.

These are the three implemented examples in the upstream PyTorch notebook. DQN/Atari and PPO appeared in its opening outline but had no runnable implementations there; they are compared conceptually at the end, not presented as implemented or verified here. No TensorFlow or Atari ROMs are required.

## 1. Environment

Install missing packages inside Colab, preserving the installed PyTorch stack. Gymnasium replaces the old Gym API. Rendering uses RGB arrays with pygame's dummy video driver: no X server, virtual display, or system installation is required. If previous experiments changed packages, start a fresh runtime.

In [ ]:
import importlib.util, subprocess, sys, os
packages={"torch":"torch","numpy":"numpy","pandas":"pandas","matplotlib":"matplotlib","PIL":"pillow"}
missing=[p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if importlib.util.find_spec("gymnasium") is None or importlib.util.find_spec("pygame") is None:
    missing.append("gymnasium[classic-control]>=1.0,<2")
if missing: subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])
os.environ["SDL_VIDEODRIVER"]="dummy"
os.environ["SDL_AUDIODRIVER"]="dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"]="1"

In [ ]:
import json, platform, random, time, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as gym
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env
from PIL import Image
from IPython.display import display, Image as DisplayImage
%matplotlib inline
SEED=42
QUICK_RUN=True
DEVICE_PREFERENCE="auto"
assert DEVICE_PREFERENCE in {"auto","cpu"}
device=torch.device("cuda" if DEVICE_PREFERENCE=="auto" and torch.cuda.is_available() else "cpu")
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(SEED)
torch.set_num_threads(min(2,torch.get_num_threads()))
OUTPUT_DIR=Path("outputs/tutorial_12");OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
versions={"python":platform.python_version(),"torch":str(torch.__version__),"gymnasium":gym.__version__,
          "numpy":np.__version__,"pandas":pd.__version__,"matplotlib":matplotlib.__version__,
          "device":str(device),"seed":SEED,"quick_run":QUICK_RUN}
display(pd.Series(versions,name="Environment"))
run_started=time.perf_counter();FIGURES=[];checks={}
EVAL_SEEDS=list(range(10000,10010 if QUICK_RUN else 10030))
def tensor(x):return torch.as_tensor(x,dtype=torch.float32,device=device)
def finish(fig,name):
    fig.tight_layout();fig.savefig(OUTPUT_DIR/name,bbox_inches="tight");FIGURES.append(name);plt.show()
def snapshot(model):return {k:v.detach().clone() for k,v in model.state_dict().items()}
def same(model,state):return all(torch.equal(v,state[k]) for k,v in model.state_dict().items())
def freeze(model,flag=True):
    for p in model.parameters():p.requires_grad_(not flag)

## 2. The environment contract and bootstrap mask

`reset(seed=...)` returns `(observation, info)`. `step(action)` returns `(next_observation, reward, terminated, truncated, info)`. Reset after **either** flag; never step a finished episode. For a continuing-task value target, bootstrap across a time limit but not a true terminal state:

`target = reward + gamma * (1 − terminated) * next_value`.

Use the final observation returned by `step`, not the reset observation. The tests below cover ordinary transitions, true termination, time-limit truncation, and both flags. A finite-horizon objective would instead require time in the observation and a different terminal-value convention.

CartPole-v1 observes cart position/velocity and pole angle/angular velocity, with two discrete pushes. Its default time limit is 500 steps. Pendulum-v1 observes cosine/sine of angle and angular velocity, uses torque in [-2,2], and truncates at 200 steps. Its rewards are usually negative: larger (closer to zero) is better. We report undiscounted episode return, distinct from the discounted training target.

In [ ]:
GAMMA=.99
def bootstrap_target(reward,next_value,terminated,gamma=GAMMA):
    return reward+gamma*(1-terminated.float())*next_value
r=torch.tensor([1.,1.,1.,1.]);v=torch.tensor([10.,10.,10.,10.])
term=torch.tensor([False,True,False,True]);trunc=torch.tensor([False,False,True,True])
target=bootstrap_target(r,v,term)
torch.testing.assert_close(target,torch.tensor([10.9,1.,10.9,1.]))
assert (term|trunc).tolist()==[False,True,True,True]
checks["termination_truncation_targets"]=True
for env_id in ["CartPole-v1","Pendulum-v1"]:
    e=gym.make(env_id);obs,info=e.reset(seed=SEED);e.action_space.seed(SEED)
    result=e.step(e.action_space.sample())
    assert len(result)==5 and e.observation_space.contains(result[0])
    print(env_id,"observation",obs.shape,"action",e.action_space,"limit",e.spec.max_episode_steps)
    e.close()

def evaluate(factory,policy,seeds=EVAL_SEEDS):
    rows=[]
    for seed in seeds:
        env=factory();obs,_=env.reset(seed=seed);rng=np.random.default_rng(seed+50000)
        total=0.;steps=0
        while True:
            action=policy(obs,rng);assert env.action_space.contains(action)
            obs,reward,terminated,truncated,info=env.step(action)
            total+=float(reward);steps+=1
            if terminated or truncated:break
            assert steps<1000,"Unexpected episode length"
        rows.append({"seed":seed,"return":total,"steps":steps,"terminated":bool(terminated),"truncated":bool(truncated)})
        env.close()
    return pd.DataFrame(rows)
def compare(factory,policies,name,title):
    result=pd.concat([evaluate(factory,p).assign(policy=label) for label,p in policies.items()],ignore_index=True)
    summary=result.groupby("policy")["return"].agg(["mean","std","min","max"])
    display(summary)
    fig,ax=plt.subplots(figsize=(8,3))
    for i,label in enumerate(policies):
        vals=result.loc[result.policy==label,"return"].to_numpy()
        ax.scatter(np.full(len(vals),i)+np.linspace(-.12,.12,len(vals)),vals,alpha=.65)
        ax.plot([i-.2,i+.2],[vals.mean()]*2,color="black",lw=2)
    ax.set(xticks=range(len(policies)),xticklabels=list(policies),ylabel="Undiscounted return",title=title)
    finish(fig,name)
    return result

## 3. CartPole: stochastic actor and learned critic

A shared MLP returns categorical action logits and a scalar V(s). Collect a complete rollout under the current policy; recursively compute discounted returns, starting from zero at true termination and V(final observation) at a time limit. For a non-truncated episode this is a Monte Carlo return with a learned baseline; at truncation it is bootstrapped. It is not one-step TD actor–critic.

Actor loss is `−mean(log π(a|s) × detached standardized advantage)` minus an entropy bonus. The critic regresses **raw discounted returns** using smooth L1 loss. Only actor advantages are standardized; do not normalize the critic's targets separately each episode. Advantages are detached so the actor loss cannot train the value head. Evaluate the final policy greedily on separate seeds; evaluation does not affect training or checkpoint selection.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__();self.shared=nn.Sequential(nn.Linear(4,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh())
        self.actor=nn.Linear(64,2);self.critic=nn.Linear(64,1)
    def forward(self,x):
        h=self.shared(x);return self.actor(h),self.critic(h).squeeze(-1)
seed_all(SEED);ac=ActorCritic().to(device);initial_ac=copy.deepcopy(ac).eval()
ac_opt=torch.optim.Adam(ac.parameters(),lr=.003)
AC_EPISODES=240 if QUICK_RUN else 1000
ac_env=gym.make("CartPole-v1");ac_history=[]
for episode in range(AC_EPISODES):
    obs,_=ac_env.reset(seed=SEED if episode==0 else None)
    logps=[];values=[];rewards=[];entropies=[]
    while True:
        logits,value=ac(tensor(obs));dist=Categorical(logits=logits);action=dist.sample()
        next_obs,reward,terminated,truncated,_=ac_env.step(int(action.item()))
        logps.append(dist.log_prob(action));values.append(value);entropies.append(dist.entropy());rewards.append(float(reward))
        obs=next_obs
        if terminated or truncated:break
    with torch.no_grad():tail=0. if terminated else ac(tensor(obs))[1].item()
    returns=[]
    for reward in reversed(rewards):tail=reward+GAMMA*tail;returns.append(tail)
    returns=tensor(list(reversed(returns)));values=torch.stack(values)
    advantage=(returns-values).detach()
    advantage=(advantage-advantage.mean())/(advantage.std(unbiased=False)+1e-8)
    actor_loss=-(torch.stack(logps)*advantage).mean()-.01*torch.stack(entropies).mean()
    critic_loss=F.smooth_l1_loss(values,returns)
    if episode==0:
        grad=torch.autograd.grad(actor_loss,list(ac.critic.parameters()),allow_unused=True,retain_graph=True)
        assert all(g is None for g in grad);checks["actor_advantage_detached"]=True
    loss=actor_loss+.5*critic_loss;assert torch.isfinite(loss)
    ac_opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(ac.parameters(),1.);ac_opt.step()
    ac_history.append({"episode":episode+1,"return":sum(rewards),"actor_loss":actor_loss.item(),"critic_loss":critic_loss.item()})
    if (episode+1)%40==0:print("CartPole episode",episode+1,"last-20 training return",np.mean([r["return"] for r in ac_history[-20:]]))
ac_env.close();ac.eval()
def ac_policy(model):
    def policy(obs,rng):
        with torch.no_grad():return int(model(tensor(obs))[0].argmax().item())
    return policy
cart_results=compare(lambda:gym.make("CartPole-v1"),
    {"Random":lambda o,r:int(r.integers(2)),"Initial greedy":ac_policy(initial_ac),"Trained greedy":ac_policy(ac)},
    "cartpole_evaluation.png","CartPole: matched evaluation seeds, final policy")
fig,axes=plt.subplots(1,2,figsize=(10,3));h=pd.DataFrame(ac_history)
axes[0].plot(h.episode,h["return"],alpha=.3,label="Episode");axes[0].plot(h.episode,h["return"].rolling(20,min_periods=1).mean(),label="20-episode mean")
axes[0].set(xlabel="Training episode",ylabel="Return");axes[0].legend()
axes[1].plot(h.episode,h.critic_loss);axes[1].set(xlabel="Training episode",ylabel="Critic smooth L1 loss")
finish(fig,"cartpole_training.png")

## 4. DDPG: replay, deterministic actor, and target networks

The actor outputs bounded continuous actions. The critic Q(s,a) estimates discounted return. Replay stores `(s,a,r,s′,terminated,truncated)`; only `terminated` suppresses bootstrapping. Targets are copied initially, frozen, and updated as `target ← (1−tau) target + tau online`.

Critic loss is mean squared TD error. Actor loss is `−mean(Q(s,actor(s)))`. Freeze critic parameters for this phase but retain gradients through its action input. We use independent Gaussian action noise scaled to the action range (instead of upstream OU noise), uniform random warmup, and one update every two environment steps. These are explicit implementation choices, not TD3: no twin critics, target smoothing, or delayed actor updates. No convergence is promised for this short DDPG run.

In [ ]:
class Replay:
    def __init__(self,obs_dim,act_dim,capacity=50000,seed=SEED):
        self.capacity=capacity;self.size=0;self.pos=0;self.rng=np.random.default_rng(seed)
        self.data={k:np.zeros((capacity,d),np.float32) for k,d in
                   [("s",obs_dim),("a",act_dim),("r",1),("next",obs_dim),("term",1),("trunc",1)]}
    def add(self,s,a,r,next_s,term,trunc):
        for k,v in zip(self.data,[s,a,r,next_s,term,trunc]):self.data[k][self.pos]=v
        self.pos=(self.pos+1)%self.capacity;self.size=min(self.size+1,self.capacity)
    def sample(self,batch=64):
        ids=self.rng.integers(self.size,size=batch)
        return {k:tensor(v[ids]) for k,v in self.data.items()}
class ContinuousActor(nn.Module):
    def __init__(self,obs_dim,low,high):
        super().__init__();self.net=nn.Sequential(nn.Linear(obs_dim,64),nn.ReLU(),nn.Linear(64,64),nn.ReLU(),nn.Linear(64,len(low)),nn.Tanh())
        self.register_buffer("scale",torch.tensor((np.array(high)-np.array(low))/2,dtype=torch.float32))
        self.register_buffer("bias",torch.tensor((np.array(high)+np.array(low))/2,dtype=torch.float32))
        nn.init.uniform_(self.net[-2].weight,-.003,.003);nn.init.uniform_(self.net[-2].bias,-.003,.003)
    def forward(self,s):return self.bias+self.scale*self.net(s)
class QNetwork(nn.Module):
    def __init__(self,obs_dim,act_dim):
        super().__init__();self.net=nn.Sequential(nn.Linear(obs_dim+act_dim,64),nn.ReLU(),nn.Linear(64,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self,s,a):return self.net(torch.cat([s,a],dim=-1))
@torch.no_grad()
def soft_update(target,online,tau=.005):
    for tp,p in zip(target.parameters(),online.parameters()):tp.mul_(1-tau).add_(p,alpha=tau)
# Replay wraparound and both ending flags remain distinct.
rb_test=Replay(1,1,capacity=3)
for i in range(5):rb_test.add([i],[0],i,[i+1],i==3,i==4)
assert rb_test.size==3 and set(rb_test.data["s"].ravel())=={2,3,4}
assert rb_test.data["term"].sum()==1 and rb_test.data["trunc"].sum()==1
checks["replay_wraparound_and_flags"]=True

def train_ddpg(factory,steps,seed):
    seed_all(seed);env=factory();obs,_=env.reset(seed=seed);env.action_space.seed(seed)
    od,ad=env.observation_space.shape[0],env.action_space.shape[0]
    actor=ContinuousActor(od,env.action_space.low,env.action_space.high).to(device)
    critic=QNetwork(od,ad).to(device);initial=copy.deepcopy(actor).eval()
    at,qt=copy.deepcopy(actor),copy.deepcopy(critic);freeze(at);freeze(qt)
    ao=torch.optim.Adam(actor.parameters(),lr=.001);qo=torch.optim.Adam(critic.parameters(),lr=.001)
    replay=Replay(od,ad,seed=seed);rng=np.random.default_rng(seed)
    history=[];losses=[];total=0.;episode=0;updates=0
    for step in range(1,steps+1):
        if step<=500:action=env.action_space.sample()
        else:
            with torch.no_grad():action=actor(tensor(obs)).cpu().numpy()
            action=np.clip(action+rng.normal(size=ad)*.1*(env.action_space.high-env.action_space.low)/2,
                           env.action_space.low,env.action_space.high).astype(np.float32)
        assert env.action_space.contains(action)
        next_obs,reward,term,trunc,_=env.step(action);replay.add(obs,action,reward,next_obs,term,trunc)
        total+=float(reward);obs=next_obs
        if step>500 and step%2==0:
            b=replay.sample();audit=updates==0
            if audit:a0,q0,at0,qt0=snapshot(actor),snapshot(critic),snapshot(at),snapshot(qt)
            with torch.no_grad():y=bootstrap_target(b["r"],qt(b["next"],at(b["next"])),b["term"])
            assert not y.requires_grad
            qloss=F.mse_loss(critic(b["s"],b["a"]),y);assert torch.isfinite(qloss)
            qo.zero_grad();qloss.backward();nn.utils.clip_grad_norm_(critic.parameters(),10.);qo.step()
            if audit:assert same(actor,a0) and not same(critic,q0)
            freeze(critic);qo.zero_grad(set_to_none=True);ao.zero_grad(set_to_none=True)
            if audit:q1=snapshot(critic)
            aloss=-critic(b["s"],actor(b["s"])).mean();assert torch.isfinite(aloss)
            aloss.backward()
            if audit:
                assert all(p.grad is None for p in critic.parameters())
                assert sum(p.grad.abs().sum().item() for p in actor.parameters())>0
            ao.step();freeze(critic,False)
            if audit:
                assert same(critic,q1) and not same(actor,a0) and same(at,at0) and same(qt,qt0)
                assert all(p.grad is None for p in at.parameters()) and all(p.grad is None for p in qt.parameters())
            soft_update(at,actor);soft_update(qt,critic)
            if audit:
                for k,p in at.named_parameters():torch.testing.assert_close(p,.995*at0[k]+.005*actor.state_dict()[k])
                checks[f"ddpg_isolation_and_soft_update_seed_{seed}"]=True
            updates+=1
            if updates%100==0:losses.append({"step":step,"critic_MSE":qloss.item(),"actor_loss":aloss.item()})
        if term or trunc:
            episode+=1;history.append({"episode":episode,"step":step,"return":total,"terminated":bool(term),"truncated":bool(trunc)})
            obs,_=env.reset();total=0.
        if step%1000==0:print("DDPG seed",seed,"step",step,"recent returns",[round(x["return"],1) for x in history[-3:]])
    env.close();actor.eval();critic.eval()
    return {"actor":actor,"critic":critic,"actor_target":at,"critic_target":qt,"initial":initial,
            "history":history,"losses":losses,"steps":steps,"updates":updates,"replay_size":replay.size}
def continuous_policy(model):
    def policy(obs,rng):
        with torch.no_grad():return model(tensor(obs)).cpu().numpy().astype(np.float32)
    return policy

In [ ]:
pend_factory=lambda:gym.make("Pendulum-v1")
pend=train_ddpg(pend_factory,6000 if QUICK_RUN else 40000,SEED+1)
pend_results=compare(pend_factory,{"Random":lambda o,r:r.uniform(-2,2,size=1).astype(np.float32),
    "Zero torque":lambda o,r:np.zeros(1,np.float32),"Initial":continuous_policy(pend["initial"]),
    "DDPG":continuous_policy(pend["actor"])},"pendulum_evaluation.png","Pendulum: noise-free policies, matched evaluation seeds")
def ddpg_plot(run,name,title):
    h=pd.DataFrame(run["history"]);l=pd.DataFrame(run["losses"]);fig,axes=plt.subplots(1,2,figsize=(10,3))
    axes[0].plot(h.step,h["return"],label="Exploratory training return")
    axes[0].plot(h.step,h["return"].rolling(5,min_periods=1).mean(),label="5-episode mean")
    axes[0].set(xlabel="Environment steps",ylabel="Return");axes[0].legend()
    axes[1].plot(l.step,l.critic_MSE);axes[1].set(xlabel="Environment steps",ylabel="Sampled critic TD MSE")
    fig.suptitle(title);finish(fig,name)
ddpg_plot(pend,"pendulum_training.png","Training returns include exploration; evaluation returns do not")

## 5. A constrained toy materials environment

Three component fractions must stay nonnegative and sum to one. A continuous action proposes a bounded adjustment; project the proposed composition onto the probability simplex. The reward is **negative Euclidean distance** to the arbitrary target (0.7,0.2,0.1), not a measured material property. The action bound constrains the proposal; the executed composition also reflects projection.

Use Gymnasium's seeded reset and checker. Fifty-step episodes end by truncation, consistent with the continuing-task bootstrap convention. The target is fixed and known: the analytic controller that moves toward it is an informative baseline, and RL is unnecessary for this toy problem. Real materials optimization needs validated property models, experimental constraints, and uncertainty handling. This section demonstrates an interface and constrained control, not material discovery.

DDPG supplies a valid continuous actor objective, replacing the upstream placeholder that treated deterministic outputs as if they were action probabilities.

In [ ]:
def simplex_projection(v):
    u=np.sort(np.asarray(v,dtype=np.float64))[::-1];cssv=np.cumsum(u)-1
    rho=np.where(u-cssv/np.arange(1,len(u)+1)>0)[0][-1]
    return np.maximum(v-cssv[rho]/(rho+1),0).astype(np.float32)
class MaterialsEnv(gym.Env):
    metadata={"render_modes":[]}
    def __init__(self):
        self.observation_space=spaces.Box(0.,1.,shape=(3,),dtype=np.float32)
        self.action_space=spaces.Box(-.1,.1,shape=(3,),dtype=np.float32)
        self.target=np.array([.7,.2,.1],np.float32);self.steps=0
    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed);self.state=self.np_random.dirichlet(np.ones(3)).astype(np.float32);self.steps=0
        return self.state.copy(),{}
    def step(self,action):
        action=np.asarray(action,dtype=np.float32);assert self.action_space.contains(action)
        self.state=simplex_projection(self.state+action);self.steps+=1
        distance=float(np.linalg.norm(self.state-self.target))
        return self.state.copy(),-distance,False,self.steps>=50,{"distance":distance}
check_env(MaterialsEnv(),skip_render_check=True)
e=MaterialsEnv();o,_=e.reset(seed=SEED);e.action_space.seed(SEED)
for _ in range(50):
    o,r,term,trunc,info=e.step(e.action_space.sample())
    assert e.observation_space.contains(o) and abs(float(o.sum())-1)<1e-6 and not term
assert trunc;e.close();checks["materials_env_checker_and_simplex"]=True
mat_factory=lambda:MaterialsEnv()
mat=train_ddpg(mat_factory,4000 if QUICK_RUN else 20000,SEED+2)
def analytic_policy(obs,rng):return np.clip(np.array([.7,.2,.1],np.float32)-obs,-.1,.1).astype(np.float32)
mat_policies={"Random":lambda o,r:r.uniform(-.1,.1,size=3).astype(np.float32),
    "No adjustment":lambda o,r:np.zeros(3,np.float32),"DDPG":continuous_policy(mat["actor"]),"Analytic":analytic_policy}
mat_results=compare(mat_factory,mat_policies,"materials_evaluation.png","Synthetic composition task: RL versus a known-target controller")
ddpg_plot(mat,"materials_training.png","Materials DDPG: synthetic reward, not a physical property")

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,3))
for label in ["DDPG","Analytic"]:
    env=MaterialsEnv();obs,_=env.reset(seed=EVAL_SEEDS[0]);path=[obs];distances=[np.linalg.norm(obs-env.target)]
    for step in range(50):
        obs,_,_,_,info=env.step(mat_policies[label](obs,np.random.default_rng(0)));path.append(obs);distances.append(info["distance"])
    path=np.array(path)
    if label=="DDPG":
        for i in range(3):
            line=axes[0].plot(path[:,i],label=f"Component {i+1}")[0]
            axes[0].axhline(env.target[i],ls="--",color=line.get_color(),alpha=.5)
    axes[1].plot(distances,label=label);env.close()
axes[0].set(xlabel="Step",ylabel="Fraction",ylim=(0,1),title="DDPG fractions; dashed = target")
axes[1].set(xlabel="Step",ylabel="Distance to target",title="First predetermined evaluation seed")
for ax in axes:ax.legend()
finish(fig,"materials_trajectory.png")

## 6. Render actual policy rollouts

The following GIFs use the first predetermined evaluation seed, not the best episode. Actions are greedy for CartPole and noise-free for Pendulum. RGB rendering is separate from training and evaluation. Store one frame every four simulation steps plus the final frame, resize for display, and read the GIF back to check it is nonempty. An animation illustrates one trajectory and cannot establish mean policy performance.

In [ ]:
def render_policy(env_id,policy,name):
    env=gym.make(env_id,render_mode="rgb_array");obs,_=env.reset(seed=EVAL_SEEDS[0]);rng=np.random.default_rng(60000)
    frames=[];total=0.;steps=0
    while True:
        if steps%4==0:frames.append(Image.fromarray(env.render()).resize((300,200)))
        obs,reward,term,trunc,_=env.step(policy(obs,rng));total+=float(reward);steps+=1
        if term or trunc:
            frames.append(Image.fromarray(env.render()).resize((300,200)));break
    env.close();frames[0].save(OUTPUT_DIR/name,save_all=True,append_images=frames[1:],duration=80 if env_id.startswith("Cart") else 200,loop=0)
    with Image.open(OUTPUT_DIR/name) as im:
        assert im.n_frames>=2;stored_frames=im.n_frames
    display(DisplayImage(filename=str(OUTPUT_DIR/name)))
    return {"seed":EVAL_SEEDS[0],"steps":steps,"return":total,"gif_frames":stored_frames}
animations={"cartpole":render_policy("CartPole-v1",ac_policy(ac),"cartpole_policy.gif"),
            "pendulum":render_policy("Pendulum-v1",continuous_policy(pend["actor"]),"pendulum_policy.gif")}
checks["rgb_render_and_gif_roundtrip"]=True
print(json.dumps(animations,indent=2))

## 7. Portable checkpoints, evaluation tables, and interpretation

Save the final policies, critics, and target networks. Reload on CPU and compare fixed-input outputs and resulting actions. Files are inference/inspection checkpoints: optimizer, replay, and RNG state are not saved, so they do not support exact training continuation. No model was chosen by evaluation performance.

Report every evaluation episode plus mean and standard deviation; these describe starting-state variability for **one trained agent**. They do not measure variability across training seeds. A short improvement over random does not establish a solved task. We do not reuse CartPole-v0's old 195-return criterion for CartPole-v1 or infer success from a single GIF.

In [ ]:
reload_checks={}
probe_cart=tensor([[0.,0.,.05,0.],[.1,-.2,-.1,.3]])
torch.save({"state_dict":{k:v.cpu() for k,v in ac.state_dict().items()}},OUTPUT_DIR/"cartpole_actor_critic.pt")
restored_ac=ActorCritic().cpu().eval();restored_ac.load_state_dict(torch.load(OUTPUT_DIR/"cartpole_actor_critic.pt",map_location="cpu",weights_only=True)["state_dict"])
with torch.no_grad():
    for a,b in zip(ac(probe_cart),restored_ac(probe_cart.cpu())):torch.testing.assert_close(a.cpu(),b,rtol=1e-4,atol=1e-5)
reload_checks["cartpole_actor_critic"]=True
for prefix,run,factory in [("pendulum",pend,pend_factory),("materials",mat,mat_factory)]:
    env=factory();obs,_=env.reset(seed=99999);od,ad=env.observation_space.shape[0],env.action_space.shape[0]
    state=tensor(np.stack([obs,obs]));action=tensor(np.stack([env.action_space.low,env.action_space.high]))
    for key in ["actor","critic","actor_target","critic_target"]:
        model=run[key];filename=OUTPUT_DIR/f"{prefix}_{key}.pt"
        torch.save({"state_dict":{k:v.cpu() for k,v in model.state_dict().items()},"observation_dim":od,
                    "action_low":env.action_space.low.tolist(),"action_high":env.action_space.high.tolist()},filename)
        data=torch.load(filename,map_location="cpu",weights_only=True)
        restored=(ContinuousActor(od,data["action_low"],data["action_high"]) if "actor" in key else QNetwork(od,ad)).cpu().eval()
        restored.load_state_dict(data["state_dict"])
        with torch.no_grad():
            expected=model(state) if "actor" in key else model(state,action)
            actual=restored(state.cpu()) if "actor" in key else restored(state.cpu(),action.cpu())
        torch.testing.assert_close(expected.cpu(),actual,rtol=1e-4,atol=1e-5)
        reload_checks[f"{prefix}_{key}"]=True
    env.close()
all_results=pd.concat([cart_results.assign(task="CartPole"),pend_results.assign(task="Pendulum"),mat_results.assign(task="Materials")],ignore_index=True)
all_results.to_csv(OUTPUT_DIR/"evaluation_episodes.csv",index=False)
assert len(pd.read_csv(OUTPUT_DIR/"evaluation_episodes.csv"))==len(EVAL_SEEDS)*11
checks["evaluation_csv_roundtrip"]=True
for name,history in [("cartpole",ac_history),("pendulum",pend["history"]),("materials",mat["history"])]:
    pd.DataFrame(history).to_csv(OUTPUT_DIR/f"{name}_training.csv",index=False)
summary=all_results.groupby(["task","policy"])["return"].agg(["mean","std","min","max"]).reset_index()
metrics={"environment":versions,"cartpole_training_episodes":AC_EPISODES,"evaluation_seeds":EVAL_SEEDS,
    "evaluation":summary.to_dict(orient="records"),"ddpg":{name:{k:run[k] for k in ["steps","updates","replay_size"]} for name,run in [("pendulum",pend),("materials",mat)]},
    "animations":animations,"checkpoint_reload":reload_checks,"checks":checks,"figures":FIGURES,"seconds_after_imports":time.perf_counter()-run_started}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(metrics,indent=2)+"\n")
print("TUTORIAL_12_COMPLETE");print(json.dumps(metrics,indent=2))

## Observed quick-run results (Colab CPU, 2026-10-12)

Across ten predetermined evaluation seeds, final CartPole greedy return was **352.8 ± 52.9** versus random **25.8 ± 14.1** (mean ± sample standard deviation). Pendulum DDPG returned **−682.3 ± 230.9**, versus random **−1057.0 ± 251.9**; control remained imperfect. Materials DDPG returned **−1.939 ± 0.577**, versus random **−26.422 ± 12.233** and analytic **−0.695 ± 0.613**. The analytic controller remained better than the learned policy.

These are starting-state statistics for one training run per task, not confidence intervals or evidence of robustness across training seeds. Greedy/noise-free evaluation differs from exploratory training. The GIFs show the first predetermined seed, and are illustrative rollouts rather than additional independent evaluation samples. See the repository validation report for versions, checks, and untested modes.

## Exercises and next steps

- Change only the seed, rerun from the top, and aggregate evaluation statistics across independently trained agents. Keep evaluation seeds fixed; add new held-out seeds before final reporting if you tune hyperparameters.
- Turn off the entropy bonus or action noise. Compare learning curves and exploration, not just one return.
- Deliberately mask truncation as terminal in a separate experiment and compare targets to the explicit unit example.
- Increase the training budget with `QUICK_RUN=False`. Longer training is not guaranteed to improve DDPG.
- Replace the toy materials reward with a documented surrogate; explain the data, units, uncertainty, and constraints before calling it materials optimization.

| Method | Learned object / action type | Data use | In this notebook |
|---|---|---|---|
| Stochastic actor + V critic | Categorical policy, discrete actions | Fresh on-policy rollouts | CartPole implementation |
| DDPG | Deterministic bounded policy + Q critic, continuous actions | Off-policy replay + targets | Pendulum and materials implementations |
| DQN | Q(s,a), typically discrete actions | Off-policy replay + target Q | Conceptual extension only; no Atari experiment |
| PPO | Clipped policy-ratio objective + value model | Reuses a fresh on-policy batch for several updates | Conceptual extension only |

The small networks are CPU-compatible. GPU and larger training budgets require their own verification; published validation records state exactly what was tested.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_12/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

License information is available in the repository `LICENSE`.

References: [Gymnasium environment API](https://gymnasium.farama.org/api/env/), [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), [Pendulum-v1](https://gymnasium.farama.org/environments/classic_control/pendulum/), [DDPG explanation](https://spinningup.openai.com/en/latest/algorithms/ddpg.html), [DDPG paper](https://arxiv.org/abs/1509.02971). Environment software has its own licensing; the materials reward and target are synthetic.